# Notebook 06 — Intégration de la connaissance experte & Pipeline Cold/Warm Start

> **Objectif** : traduire la connaissance métier en contraintes formelles du modèle, et implémenter le pipeline complet cold → warm → mature.

---

## Concepts abordés
1. Priors informés bayésiens (connaissance experte = prior)
2. Contraintes sur les coefficients (Mixed Effects)
3. Pipeline cold → warm → mature : implémentation complète
4. Monitoring de la convergence (quand passer d'une phase à l'autre ?)
5. Synthèse et recommandations pour PRJ2025_773

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
FEATURES = ['composants', 'mo', 'energie', 'volume']
TARGET   = 'cout'
print(f"Dataset : {len(df):,} lignes")

## 1. Intégration de la connaissance experte

### Dans le modèle bayésien : prior informé

L'expert dit : *"Cette variante coûte +15% car elle utilise des condensateurs premium"*

**Sans connaissance experte** (prior non-informatif) :
$$\mu_v \sim \mathcal{N}(\mu_f, \sigma_f^2)$$

**Avec connaissance experte** (prior informé) :
$$\mu_v \sim \mathcal{N}(1.15 \times \mu_f, \; (0.5 \times \sigma_f)^2)$$

Le prior est centré sur +15% et a une incertitude réduite (on est plus sûr du driver).

### Comparaison quantitative

In [ ]:
def bayesian_shrinkage(y_obs, mu_prior, sigma_prior, sigma_eps):
    n        = len(y_obs)
    y_bar    = y_obs.mean()
    lambda_  = n / (n + (sigma_eps / sigma_prior) ** 2)
    mu_post  = lambda_ * y_bar + (1 - lambda_) * mu_prior
    sigma_post = np.sqrt(1 / (n / sigma_eps**2 + 1 / sigma_prior**2))
    return mu_post, sigma_post, lambda_


# Scénario : variante premium avec composant spécial
MU_FAMILLE   = 1000.0
SIGMA_FAMILLE = 120.0
SIGMA_EPS     = 80.0
VRAIE_VALEUR  = 1150.0  # +15% effectivement

# Simule 3 observations (variante rare)
y_obs = np.random.normal(VRAIE_VALEUR, SIGMA_EPS, 3)

# Prior non-informatif
mu_non_inf, sig_non_inf, _ = bayesian_shrinkage(
    y_obs, MU_FAMILLE, SIGMA_FAMILLE, SIGMA_EPS
)

# Prior informé par l'expert (+15%, incertitude réduite)
mu_expert, sig_expert, _ = bayesian_shrinkage(
    y_obs,
    mu_prior    = 1.15 * MU_FAMILLE,   # centré sur +15%
    sigma_prior = 0.5  * SIGMA_FAMILLE, # expert plus sûr de lui
    sigma_eps   = SIGMA_EPS
)

print(f"Vraie valeur         : {VRAIE_VALEUR:.0f}€")
print(f"Observations brutes  : {y_obs.mean():.0f}€ (n=3)")
print()
print(f"Prior non-informatif : {mu_non_inf:.0f}€ ± {1.96*sig_non_inf:.0f}€")
print(f"Prior expert (+15%)  : {mu_expert:.0f}€  ± {1.96*sig_expert:.0f}€")
print()
print(f"Erreur prior non-inf : {abs(mu_non_inf - VRAIE_VALEUR):.0f}€")
print(f"Erreur prior expert  : {abs(mu_expert  - VRAIE_VALEUR):.0f}€")

In [ ]:
# Visualisation des posteriors
x_range = np.linspace(700, 1400, 500)

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

fig, ax = plt.subplots(figsize=(11, 5))

# Prior non-informatif
ax.plot(x_range, normal_pdf(x_range, MU_FAMILLE, SIGMA_FAMILLE),
        'gray', linestyle=':', lw=2, label=f'Prior famille (non-inf.) — μ={MU_FAMILLE:.0f}€')

# Vraisemblance (données)
ax.plot(x_range, normal_pdf(x_range, y_obs.mean(), SIGMA_EPS / np.sqrt(3)),
        '#e74c3c', linestyle='--', lw=2, label=f'Vraisemblance (données n=3) — ȳ={y_obs.mean():.0f}€')

# Posterior non-informatif
ax.fill_between(x_range, normal_pdf(x_range, mu_non_inf, sig_non_inf),
                alpha=0.3, color='#3498db',
                label=f'Posterior (prior non-inf.) — μ_post={mu_non_inf:.0f}€')

# Posterior expert
ax.fill_between(x_range, normal_pdf(x_range, mu_expert, sig_expert),
                alpha=0.3, color='#27ae60',
                label=f'Posterior (prior expert +15%) — μ_post={mu_expert:.0f}€')

ax.axvline(VRAIE_VALEUR, color='black', linestyle='-', lw=2, label=f'Vraie valeur = {VRAIE_VALEUR}€')

ax.set_xlabel('Coût (€)')
ax.set_ylabel('Densité de probabilité')
ax.set_title('Impact du prior expert sur l\'estimation bayésienne\n(n=3 observations)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('data/fig_expert_prior.png', dpi=150)
plt.show()

## 2. Pipeline Cold → Warm → Mature : implémentation complète

On implémente un système qui, pour chaque variante, sélectionne automatiquement la bonne stratégie selon le nombre d'observations disponibles.

In [ ]:
@dataclass
class VariantCostEstimator:
    """
    Estimateur de coût pour une variante industrielle.
    Sélectionne automatiquement la stratégie selon le nombre d'observations.
    """
    mu_famille   : float
    sigma_famille: float
    sigma_eps    : float
    expert_prior : Optional[dict] = None  # {'mu_factor': 1.15, 'sigma_factor': 0.5}

    observations : list = field(default_factory=list)

    THRESHOLDS = {'cold': 3, 'warm': 6, 'semi': 12}  # seuils de phase
    LAMBDAS    = {'cold': 0.0, 'warm': 0.2, 'semi': 0.5, 'mature': 0.8}

    def add_observation(self, cost: float) -> None:
        self.observations.append(cost)

    @property
    def phase(self) -> str:
        n = len(self.observations)
        if n < self.THRESHOLDS['cold']:  return 'cold'
        if n < self.THRESHOLDS['warm']:  return 'warm'
        if n < self.THRESHOLDS['semi']:  return 'semi'
        return 'mature'

    @property
    def mu_prior(self) -> float:
        if self.expert_prior:
            return self.expert_prior['mu_factor'] * self.mu_famille
        return self.mu_famille

    @property
    def sigma_prior(self) -> float:
        if self.expert_prior:
            return self.expert_prior['sigma_factor'] * self.sigma_famille
        return self.sigma_famille

    def estimate(self) -> dict:
        n = len(self.observations)

        if n == 0:
            return {
                'phase'     : 'cold',
                'mu_est'    : self.mu_prior,
                'ic_half'   : 1.96 * self.sigma_prior,
                'lambda'    : 0.0,
                'n_obs'     : 0,
            }

        y_obs = np.array(self.observations)
        mu_post, sigma_post, lam = bayesian_shrinkage(
            y_obs, self.mu_prior, self.sigma_prior, self.sigma_eps
        )

        return {
            'phase'   : self.phase,
            'mu_est'  : mu_post,
            'ic_half' : 1.96 * sigma_post,
            'lambda'  : lam,
            'n_obs'   : n,
        }

In [ ]:
# Simulation : arrivée progressive d'observations pour une variante
VRAIE_MOYENNE = 1150.0

# Variante sans connaissance experte
est_standard = VariantCostEstimator(
    mu_famille=1000.0, sigma_famille=120.0, sigma_eps=80.0
)

# Variante avec prior expert (+15%, condensateur premium)
est_expert = VariantCostEstimator(
    mu_famille=1000.0, sigma_famille=120.0, sigma_eps=80.0,
    expert_prior={'mu_factor': 1.15, 'sigma_factor': 0.5}
)

# Simulation de l'arrivée des commandes
commandes = np.random.normal(VRAIE_MOYENNE, 80.0, 30)

timeline_std, timeline_exp = [], []

# Phase 0 : avant toute commande
timeline_std.append(est_standard.estimate())
timeline_exp.append(est_expert.estimate())

for cout in commandes:
    est_standard.add_observation(cout)
    est_expert.add_observation(cout)
    timeline_std.append(est_standard.estimate())
    timeline_exp.append(est_expert.estimate())

df_timeline = pd.DataFrame({
    'n_obs'    : [r['n_obs'] for r in timeline_std],
    'mu_std'   : [r['mu_est'] for r in timeline_std],
    'ic_std'   : [r['ic_half'] for r in timeline_std],
    'mu_exp'   : [r['mu_est'] for r in timeline_exp],
    'ic_exp'   : [r['ic_half'] for r in timeline_exp],
    'phase'    : [r['phase'] for r in timeline_std],
    'lambda'   : [r['lambda'] for r in timeline_std],
})

print("Évolution de l'estimation au fil des commandes :")
print(df_timeline[df_timeline['n_obs'].isin([0,1,3,5,10,20,30])].round(0).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Estimation au fil du temps
ax = axes[0]
x = df_timeline['n_obs'].values

ax.fill_between(x, df_timeline['mu_std'] - df_timeline['ic_std'],
                   df_timeline['mu_std'] + df_timeline['ic_std'],
                alpha=0.2, color='#3498db')
ax.plot(x, df_timeline['mu_std'], 'o-', color='#3498db', lw=2, ms=4,
        label='Estimateur (prior non-inf.)')

ax.fill_between(x, df_timeline['mu_exp'] - df_timeline['ic_exp'],
                   df_timeline['mu_exp'] + df_timeline['ic_exp'],
                alpha=0.2, color='#27ae60')
ax.plot(x, df_timeline['mu_exp'], 's-', color='#27ae60', lw=2, ms=4,
        label='Estimateur (prior expert +15%)')

ax.axhline(VRAIE_MOYENNE, color='black', linestyle='--', lw=2, label=f'Vraie valeur = {VRAIE_MOYENNE}€')
ax.axhline(1000, color='gray', linestyle=':', lw=1, label='Moyenne famille = 1000€')

# Zones de phase
phase_colors = {'cold': '#ffcccc', 'warm': '#fff3cc', 'semi': '#cce5ff', 'mature': '#ccffcc'}
for i, (ph, col) in enumerate(phase_colors.items()):
    prev = [0, 3, 6, 12][i]
    nxt  = [3, 6, 12, 31][i]
    ax.axvspan(prev, min(nxt, 30), alpha=0.15, color=col, label=f'Phase {ph}')

ax.set_xlabel("Nombre d'observations")
ax.set_ylabel('Coût estimé (€)')
ax.set_title('Convergence de l\'estimation selon le nombre d\'observations\n'
             'et l\'impact du prior expert')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(0, 30)

# Lambda (poids des données)
ax = axes[1]
ax.plot(x, df_timeline['lambda'], 'o-', color='#9b59b6', lw=2, ms=5)
ax.set_xlabel("Nombre d'observations")
ax.set_ylabel('λ (poids accordé aux données locales)')
ax.set_title('Shrinkage progressif : de 0 (prior pur) à 1 (données locales)')
ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='λ=0.5 (équilibre prior/données)')
ax.set_xlim(0, 30)
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_cold_warm_mature_pipeline.png', dpi=150)
plt.show()

## 3. Monitoring : quand changer de phase ?

La transition entre phases doit être **automatisée** avec des critères objectifs :

In [ ]:
def phase_diagnostic(observations: list, mu_famille: float, sigma_famille: float,
                      sigma_eps: float, seuil_drift: float = 0.2) -> dict:
    """
    Retourne un diagnostic de phase pour une variante.
    Critères :
    - Phase : basée sur n_obs
    - Drift : |mu_variante - mu_famille| / sigma_famille > seuil_drift => déviation significative
    - Stabilité : écart-type des dernières estimations (rolling window)
    """
    n = len(observations)
    if n == 0:
        return {'phase': 'cold', 'drift_detected': False, 'action': 'Utiliser modèle global'}

    y_obs = np.array(observations)
    mu_post, sigma_post, lam = bayesian_shrinkage(
        y_obs, mu_famille, sigma_famille, sigma_eps
    )

    drift = abs(mu_post - mu_famille) / sigma_famille
    drift_detected = drift > seuil_drift

    if n < 3:
        phase  = 'cold'
        action = f'Modèle global uniquement (IC={1.96*sigma_post:.0f}€)'
    elif n < 6:
        phase  = 'warm'
        action = f'Shrinkage fort λ={lam:.2f} (80% famille, 20% local)'
    elif n < 12:
        phase  = 'semi'
        action = f'Shrinkage modéré λ={lam:.2f}'
    else:
        phase  = 'mature'
        action = f'Modèle autonome λ={lam:.2f}, LOO-CV fiable'

    if drift_detected:
        action += f' ⚠ Drift détecté ({drift:.2f}σ) — vérifier avec expert'

    return {
        'n_obs'         : n,
        'phase'         : phase,
        'mu_estimé'     : mu_post,
        'incertitude'   : sigma_post,
        'lambda'        : lam,
        'drift'         : drift,
        'drift_detected': drift_detected,
        'action'        : action,
    }


# Démonstration sur 3 scénarios
scenarios = [
    ('Variante nouvelle (0 obs)', []),
    ('Variante rare conforme (3 obs)', list(np.random.normal(1020, 80, 3))),
    ('Variante rare dérivante (3 obs)', list(np.random.normal(1350, 80, 3))),
    ('Variante mature (20 obs)', list(np.random.normal(1100, 80, 20))),
]

for nom, obs in scenarios:
    diag = phase_diagnostic(obs, mu_famille=1000, sigma_famille=120, sigma_eps=80)
    print(f"\n--- {nom} ---")
    for k, v in diag.items():
        if isinstance(v, float): print(f"  {k:20s}: {v:.2f}")
        else: print(f"  {k:20s}: {v}")

## 4. Synthèse et recommandations pour PRJ2025_773

In [ ]:
# Vue d'ensemble du pipeline complet
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

pipeline_text = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║                   PIPELINE COSTING HIÉRARCHIQUE — PRJ2025_773                  ║
╠══════════════╦════════════════╦═════════════════╦═════════════════════════════╣
║ Phase        ║ n observations ║ Stratégie       ║ Validation                  ║
╠══════════════╬════════════════╬═════════════════╬═════════════════════════════╣
║ Cold start   ║ 0 obs          ║ Modèle global   ║ Heldout families            ║
║              ║                ║ λ = 0.0         ║ IC large (sigma_famille)    ║
╠══════════════╬════════════════╬═════════════════╬═════════════════════════════╣
║ Warm start   ║ 1-5 obs        ║ Shrinkage fort  ║ LOO-CV obligatoire          ║
║              ║                ║ λ = 0.2         ║ Prior expert si disponible  ║
╠══════════════╬════════════════╬═════════════════╬═════════════════════════════╣
║ Semi-mature  ║ 6-11 obs       ║ Shrinkage modéré║ LOO-CV + monitoring drift   ║
║              ║                ║ λ = 0.5         ║                             ║
╠══════════════╬════════════════╬═════════════════╬═════════════════════════════╣
║ Mature       ║ ≥ 12 obs       ║ Fine-tuning     ║ Walk-forward validation     ║
║              ║                ║ λ = 0.8         ║ LOO-CV fiable               ║
╚══════════════╩════════════════╩═════════════════╩═════════════════════════════╝

Approche recommandée :
  1. Démarrer avec Mixed Effects (Approche B) — lisible, rapide, communicable en CODIR
  2. Paralléliser Bayésien (Approche A) sur les familles à forte proportion de variantes rares
  3. Évaluer avec perte asymétrique (3x sur sous-estimation) + taux de couverture
"""

ax.text(0.02, 0.95, pipeline_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.8))

plt.tight_layout()
plt.savefig('data/fig_pipeline_synthese.png', dpi=150, bbox_inches='tight')
plt.show()

## Résumé final de la série

| Notebook | Concept clé appris |
|----------|-------------------|
| 00 | Génération de données synthétiques, structure hiérarchique |
| 01 | Biais-variance, détection over-fit (LOO ratio, CV prédit) |
| **04** | **Transfer Learning : cold start, fine-tuning sur résidus — point de départ** |
| 02 | Mixed Effects : couche d'explicabilité pour le CODIR |
| 03 | Bayésien hiérarchique : prior → posterior → IC formels |
| 05 | Protocole LOO-CV, heldout families, métriques asymétriques |
| **06** | **Prior expert, pipeline cold→mature, monitoring de convergence** |

---

## Recommandation finale pour PRJ2025_773

```
Étape 1 — Transfer Learning (NB 04)  [obligatoire]
  → Modèle global GBM sur toutes les variantes + features hiérarchiques
  → Valider le RMSE sur les matures (walk-forward, NB 05)
  → Cold start immédiat pour les nouvelles variantes
  → Livrable : baseline opérationnelle

Étape 2 — Mixed Effects (NB 02)  [si explicabilité nécessaire]
  → Couche "carte de coût" par famille pour le CODIR
  → Ne remplace pas le GBM, l'accompagne
  → Livrable : tableau effets famille + déviations variantes

Étape 3 — Bayésien (NB 03)  [si IC formels ou knowledge expert à intégrer]
  → Sur les familles à forte proportion de variantes rares
  → Priors informés si l'expert a des règles de coût précises
  → Livrable : intervalles de confiance sur chaque devis
```

**Critère d'arrêt** : si la performance de l'Étape 1 est satisfaisante et que le CODIR n'exige pas d'explications par famille, il n'est pas nécessaire d'aller plus loin.